# Vision ORPO

Combine the chosen-answer language loss with an odds-ratio preference objective.
ORPO uses one model.
The text-only preference dataset is loaded from Hugging Face below.


## Setup

Use an Apple Silicon Mac and the Python environment selected by your notebook kernel.
Install once from your checkout, then restart the kernel:

```sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel
```

Run the cells in order. The first load downloads the model and dataset.


In [ ]:
from pathlib import Path

import mlx.core as mx
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.trainer import ORPOTrainingArgs, Trainer, prepare_model_for_training


## 1. Settings

Edit the model, data, and output path below. `iters` counts batches;
four accumulation steps make one optimizer update. Paths use the kernel's working directory.


In [ ]:
MODEL_ID = "mlx-community/Qwen2.5-VL-3B-Instruct-4bit"
DATASET_ID = "mlx-community/Josiefied-Qwen3-dpo-v1-flat"
OUTPUT_DIR = Path("adapters/vlm/orpo")
SEED = 42

# Adapters
TRAIN_TYPE = "lora"  # "lora", "dora", or "full"
CHECKPOINT_PATH = None
LORA_RANK = 8
LORA_ALPHA = 16

# Training
args = ORPOTrainingArgs(
    iters=100,
    batch_size=1,
    learning_rate=1e-4,
    gradient_accumulation_steps=4,
    max_seq_length=2048,
    beta=0.1,
    cache_size=8,
    grad_checkpoint=True,
    compile=False,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=100,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Data

Load the text-only Hugging Face dataset. Its rows contain `prompt`, `chosen`, and `rejected` strings. The repository currently has only a `train` split, so evaluation is skipped unless a validation split is added.


In [ ]:
dataset = load_dataset(DATASET_ID)
train_rows = dataset["train"]
eval_rows = dataset.get("validation", dataset.get("valid"))

required_columns = {"prompt", "chosen", "rejected"}
missing_columns = required_columns - set(train_rows.column_names)
assert not missing_columns, f"Dataset is missing columns: {sorted(missing_columns)}"
assert len(train_rows) >= args.batch_size, (
    "Reduce batch_size or add more training rows."
)
assert eval_rows is None or len(eval_rows) >= args.batch_size, (
    "Validation needs a complete batch."
)
print(
    {
        "train_rows": len(train_rows),
        "validation_rows": 0 if eval_rows is None else len(eval_rows),
        "columns": train_rows.column_names,
    }
)


## 3. Model

Load a fresh base model before preparing its adapters.


In [ ]:
model, processor = load(MODEL_ID)

### Prepare adapters

LoRA trains the language adapters while freezing the base. `TRAIN_TYPE` also accepts
`"dora"` or `"full"`; full training unfreezes all weights and needs more memory.
Set `CHECKPOINT_PATH` to saved weights to continue with a fresh optimizer.
See the [training guide](../../docs/training.md) for quantization and target-layer options.


In [ ]:
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    checkpoint_path=CHECKPOINT_PATH,
)

## 4. Train

Each batch compares the chosen and rejected responses for the same image and prompt.


In [ ]:
trainer = Trainer(
    model,
    args,
    task="vlm",
    algorithm="orpo",
    processing_class=processor,
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    seed=SEED,
)

In [ ]:
metrics = trainer.train()

### Metrics

The table shows the last five training reports. All metrics remain in `metrics`
and `trainer.log_history`, including loss diagnostics, timing, and memory.
`trained_tokens` counts supervised targets; `total_vision_tokens` counts expanded
image/video input positions. Processing token/s includes forward, backward, and updates.


In [ ]:
report_history = pd.DataFrame(trainer.log_history)
columns = {
    "iteration": "step",
    "train_loss": "loss",
    "learning_rate": "lr",
    "trained_tokens": "trained tokens",
    "total_vision_tokens": "vision tokens",
    "processed_tokens_per_second": "processed tok/s",
    "peak_memory": "peak GB",
}
recent_reports = (
    report_history.dropna(subset=["train_loss"])
    .tail(5)
    .reindex(columns=columns)
    .rename(columns=columns)
    .set_index("step")
)
recent_reports.round({"loss": 4, "processed tok/s": 1, "peak GB": 2})

## 5. Evaluate

Check preference loss on the validation pairs. Replace the toy rows with held-out
examples from your task to assess useful preference learning.


In [ ]:
assert mx.isfinite(mx.array(metrics["train_loss"])).item(), "Non-finite training loss."
validation = trainer.evaluate() if eval_rows is not None else {}
pd.DataFrame([validation]).filter(
    items=["val_loss", "val_preference_accuracy", "val_odds_margin"]
).round(4)

## 6. Save

Keep `weights.safetensors` and `adapter_config.json` together. The loop also saves
the final checkpoint automatically.


In [ ]:
trainer.save_adapter()
print(Path(args.adapter_file).resolve())

### Reload later

Load the original base model, then attach the saved weights:

```python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
model.eval()
```

More options: [training guide](../../docs/training.md).
